# 02. labeled_data 상관계수와 부품·좌우 구조

[분석 보고서](../artifacts/correlation_v1/report.md)에 차종 코드, JX1/SP2 처리, LH/RH와 캐비티 수에 관한 근거가 있습니다. **모델 학습이 아닌 탐색 분석**입니다. `Python (kamp-base)` 커널에서 위에서 아래로 실행합니다.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json, subprocess, sys
from datetime import datetime
from IPython.display import display, Image
PROJECT = None
for base in [Path.cwd(), *Path.cwd().parents]:
    for candidate in [base, base / "1.injection-molding"]:
        if (candidate / "scripts/labeled_correlations.py").exists():
            PROJECT = candidate
            break
    if PROJECT is not None:
        break
if PROJECT is None:
    raise FileNotFoundError("사출 프로젝트 폴더에서 열어 주세요.")
RESULTS = PROJECT / "artifacts/correlation_v1"
RUN_ANALYSIS = False
if RUN_ANALYSIS:
    RESULTS = PROJECT / "artifacts/correlation_runs" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    subprocess.run([sys.executable, str(PROJECT / "scripts/labeled_correlations.py"), "--output", str(RESULTS)], check=True)
print(RESULTS)

## 1. JX1·SP2를 포함한 실제 데이터 수

JX1은 GV80, SP2는 셀토스에 대응하는 차종 코드입니다. 사출기 모델명이 아닙니다. JX1/SP2는 각각 한 기록이 두 번 반복되어 고유 관측이 1개씩이고 모두 양품입니다. 각각 S12/S01에서 생산된 다른 제품입니다. 원본은 보존하고 현재 S14 CN7·RG3 분석 대상에서는 제외합니다. SP2의 루프랙 커버를 앞유리 몰딩으로 합치지 않습니다.

In [ ]:
display(pd.read_csv(RESULTS / "product_inventory.csv"))
display(pd.read_csv(RESULTS / "cohort_summary.csv"))

## 2. 공정 열끼리의 상관계수

**Pearson**: 값 사이 선형 관계. 극단값 영향이 클 수 있습니다. **Spearman**: 순위로 계산하는 단조 관계. 서로 다른 특징을 보므로 둘 다 봅니다.

+1은 함께 증가, -1은 반대로 움직이는 관계입니다. 0은 해당 방식에서 뚜렷한 관계가 없다는 뜻이며 모든 비선형 관계가 없다는 뜻은 아닙니다. 상관이 높아도 원인 관계나 불량 예측력을 뜻하지 않습니다.

아래 기본값은 제품별 분석입니다. 전체 혼합(`all_products_reference`, `s14_pooled_reference`)은 제품/설비 간 차이가 섞인 참고 자료입니다.

In [ ]:
COHORT = "cn7"  # cn7, rg3, cn7_lh, cn7_rh, rg3_lh, rg3_rh
METHOD = "spearman"  # pearson 또는 spearman
matrix = pd.read_csv(RESULTS / f"matrices/{COHORT}_{METHOD}.csv", index_col=0)
pairs = pd.read_csv(RESULTS / "feature_pairs.csv")
selected = pairs[pairs.cohort.eq(COHORT) & pairs.method.eq(METHOD)].copy()
selected["absolute_correlation"] = selected.correlation.abs()
display(selected.sort_values("absolute_correlation", ascending=False).head(15))
figure = RESULTS / f"figures/{COHORT}_{METHOD}.png"
if figure.exists():
    display(Image(filename=str(figure), width=1000))
else:
    display(matrix.round(3))

## 3. 상수 열은 상관계수 0이 아닙니다

변하지 않는 열은 분산이 0이므로 상관을 정의할 수 없습니다. CSV에는 빈 값(NaN)으로 남기고 히트맵에서는 제외했습니다. ID·생산일련번호·설비코드는 연속 공정값이 아니므로 입력 열에서 제외했습니다.

In [ ]:
constant = pd.read_csv(RESULTS / "constant_columns.csv")
display(constant[constant.cohort.eq(COHORT)])

## 4. 불량(1)과의 상관은 따로 보기

아래 표는 공정값–불량 라벨의 관계입니다. `pearson_within_day_side`는 날짜·좌우별 평균을 양쪽 변수에서 빼고 계산한 민감도 분석입니다. 날짜별 운전 조건 차이를 일부 줄이지만 인과 효과는 아닙니다.

불량 수가 적고 시간·좌우 관측이 독립적이지 않아 p-value는 제시하지 않습니다. 이 표의 값은 정확도나 Recall도 아닙니다. 표를 보고 변수를 선택한 모델은 별도 미래 검증이 필요합니다.

In [ ]:
targets = pd.read_csv(RESULTS / "target_correlations.csv")
t = targets[targets.cohort.eq(COHORT)].pivot(index="feature", columns="method", values="correlation_with_defect")
t = t.reindex(t["pearson"].abs().sort_values(ascending=False).index)
display(t.round(4))
display(pd.read_csv(RESULTS / "cohort_summary.csv").query("cohort == @COHORT"))

## 5. 좌우가 같은 공정 이벤트인가? 캐비티는 몇 개인가?

LH/RH는 차량 진행 방향 기준 왼쪽/오른쪽입니다. 캐비티는 사출기 자체가 아니라 **장착된 금형의 제품 형상 공간**입니다.

동일 설비·시각의 좌우 공정값이 같은 쌍은 많이 있지만, 이것만으로 2캐비티를 확정할 수 없습니다. 실제 동시 생산뿐 아니라 설비 로그를 좌우 검사에 연결한 기록 구조로도 이런 표가 나올 수 있습니다. 금형 ID·캐비티 수·샷 번호·취출 수 확인이 필요합니다.

In [ ]:
display(pd.read_csv(RESULTS / "paired_event_audit.csv"))

## 6. 같은 공정값을 좌우에서 두 번 센 영향

공정 열끼리의 관계는 같은 설비·시각·36개 공정값을 한 번만 센 표와도 비교합니다. 원본을 삭제하거나 좌우 불량 라벨을 합친 것이 아닙니다. 부품 단위 라벨 분석과 설비 이벤트 단위 공정 분석을 구분하기 위한 민감도 점검입니다.

In [ ]:
PRODUCT = "cn7"  # cn7 또는 rg3
METHOD = "pearson"
part = pd.read_csv(RESULTS / f"matrices/{PRODUCT}_{METHOD}.csv", index_col=0)
event = pd.read_csv(RESULTS / f"matrices/{PRODUCT}_event_process_{METHOD}.csv", index_col=0)
rows = []
for i, a in enumerate(part.columns):
    for b in part.columns[i+1:]:
        if pd.notna(part.loc[a,b]) and pd.notna(event.loc[a,b]):
            rows.append({"feature_a":a,"feature_b":b,"part_weighted":part.loc[a,b],"event_process_weighted":event.loc[a,b],"absolute_change":abs(part.loc[a,b]-event.loc[a,b])})
display(pd.DataFrame(rows).sort_values("absolute_change", ascending=False).head(10))

지금은 **상관을 설명하는 단계**입니다. 높은 상관의 컬럼을 자동 삭제하거나, 상관이 가장 큰 열을 불량 원인으로 확정하지 않습니다. 다음 운전 조건·센서 구간 분석과 시간 검증 설계에 참고합니다. 자세한 해석과 외부 차종 근거는 [보고서](../artifacts/correlation_v1/report.md)에 있습니다.